# Week 15 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 15 Day 01: Option payoff structure

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 07: Portfolio Project
- Previous lesson file: content/week-14/day-07.md
- Today's deliverable: Implement payoff calculator for vanilla option strategies.
- Next handoff target: Week 15 Day 02: Greeks intuition
- Next lesson file: content/week-15/day-02.md

## Theory Concepts

### Concept 1: Calls and puts
Calls and puts is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Long vs short payoff asymmetry
Long vs short payoff asymmetry is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Moneyness intuition
Moneyness intuition is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Plot payoff diagrams for basic option positions at expiry.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Option payoff structure'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement payoff calculator for vanilla option strategies.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 01): Explain Portfolio Return and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use Portfolio Return as a decision bridge from market observations to position sizing. The formula is $\mu_p=w^\top\mu$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Option payoff structure' matter in live trading systems?
   - Model answer: "Option payoff structure matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Option payoff structure in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which payoff shape best fits downside protection goals?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 15 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1501)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033984867007,
 'portfolio_volatility': 0.07641260569288279,
 'turnover_proxy': 0.2575704363752089}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10501)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Option payoff structure',
    'week': 15,
    'day': 1,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Option payoff structure',
 'week': 15,
 'day': 1,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 01 Quiz

Topic: **Option payoff structure**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [4]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Option payoff structure')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Option payoff structure
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975


# Week 15 Day 02: Greeks intuition

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 01: Option payoff structure
- Previous lesson file: content/week-15/day-01.md
- Today's deliverable: Create a Greeks approximation table for synthetic option quotes.
- Next handoff target: Week 15 Day 03: Volatility surface basics
- Next lesson file: content/week-15/day-03.md

## Theory Concepts

### Concept 1: Delta sensitivity
Delta sensitivity is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Gamma curvature
Gamma curvature is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Vega and implied volatility exposure
Vega and implied volatility exposure is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Track delta and gamma behavior near at-the-money conditions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Greeks intuition'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a Greeks approximation table for synthetic option quotes.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 02): Explain Portfolio Variance and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use Portfolio Variance as a decision bridge from market observations to position sizing. The formula is $\sigma_p^2=w^\top\Sigma w$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Greeks intuition' matter in live trading systems?
   - Model answer: "Greeks intuition matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Greeks intuition in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
How does gamma risk change near expiry?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 15 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [5]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1502)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033984867007,
 'portfolio_volatility': 0.07641260569288279,
 'turnover_proxy': 0.2575704363752089}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [6]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10502)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Greeks intuition',
    'week': 15,
    'day': 2,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Greeks intuition',
 'week': 15,
 'day': 2,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 02 Quiz

Topic: **Greeks intuition**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [7]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Greeks intuition')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Greeks intuition
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975


# Week 15 Day 03: Volatility surface basics

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 02: Greeks intuition
- Previous lesson file: content/week-15/day-02.md
- Today's deliverable: Build implied-vol surface toy visualization.
- Next handoff target: Week 15 Day 04: Hedging workflow basics
- Next lesson file: content/week-15/day-04.md

## Theory Concepts

### Concept 1: Implied volatility
Implied volatility is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Smile/skew interpretation
Smile/skew interpretation is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Vol regime context
Vol regime context is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Simulate implied-volatility skew and discuss directional implications.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Volatility surface basics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build implied-vol surface toy visualization.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 03): Explain Risk Contribution and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use Risk Contribution as a decision bridge from market observations to position sizing. The formula is $RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Volatility surface basics' matter in live trading systems?
   - Model answer: "Volatility surface basics matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Volatility surface basics in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Why does skew differ across equities and indices?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 15 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1503)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033984867007,
 'portfolio_volatility': 0.07641260569288279,
 'turnover_proxy': 0.2575704363752089}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [9]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10503)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Volatility surface basics',
    'week': 15,
    'day': 3,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Volatility surface basics',
 'week': 15,
 'day': 3,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 03 Quiz

Topic: **Volatility surface basics**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [10]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Volatility surface basics')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Volatility surface basics
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975


# Week 15 Day 04: Hedging workflow basics

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 03: Volatility surface basics
- Previous lesson file: content/week-15/day-03.md
- Today's deliverable: Implement discrete hedging simulation and PnL decomposition.
- Next handoff target: Week 15 Day 05: Options strategy framing
- Next lesson file: content/week-15/day-05.md

## Theory Concepts

### Concept 1: Delta hedging intuition
Delta hedging intuition is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Rehedging frequency tradeoff
Rehedging frequency tradeoff is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Hedging costs and slippage
Hedging costs and slippage is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Simulate a simple delta-hedging loop on synthetic price paths.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Hedging workflow basics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement discrete hedging simulation and PnL decomposition.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 04): Explain Duration Shock and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use Duration Shock as a decision bridge from market observations to position sizing. The formula is $\frac{\Delta P}{P}\approx-D_{mod}\Delta y$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Hedging workflow basics' matter in live trading systems?
   - Model answer: "Hedging workflow basics matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Hedging workflow basics in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
When can hedging itself create risk concentration?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 15 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [11]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1504)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033984867007,
 'portfolio_volatility': 0.07641260569288279,
 'turnover_proxy': 0.2575704363752089}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [12]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10504)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Hedging workflow basics',
    'week': 15,
    'day': 4,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Hedging workflow basics',
 'week': 15,
 'day': 4,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 04 Quiz

Topic: **Hedging workflow basics**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [13]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Hedging workflow basics')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Hedging workflow basics
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975


# Week 15 Day 05: Options strategy framing

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Build options intuition for payoffs, Greeks, and hedging workflows.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 04: Hedging workflow basics
- Previous lesson file: content/week-15/day-04.md
- Today's deliverable: Build strategy scenario matrix with payoff summaries.
- Next handoff target: Week 15 Day 06: Revision Sprint
- Next lesson file: content/week-15/day-06.md

## Theory Concepts

### Concept 1: Directional vs volatility bets
Directional vs volatility bets is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Risk-defined structures
Risk-defined structures is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Scenario-based comparison
Scenario-based comparison is a core part of 'Derivatives and options basics'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Compare covered call, protective put, and straddle under scenarios.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Options strategy framing'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build strategy scenario matrix with payoff summaries.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 05): Explain CVaR and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use CVaR as a decision bridge from market observations to position sizing. The formula is $CVaR_\alpha=E[L\mid L\ge VaR_\alpha]$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Options strategy framing' matter in live trading systems?
   - Model answer: "Options strategy framing matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Options strategy framing in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which strategy aligns with your current risk tolerance and why?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Week 15 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [14]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1505)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033984867007,
 'portfolio_volatility': 0.07641260569288279,
 'turnover_proxy': 0.2575704363752089}

## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [15]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10505)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Options strategy framing',
    'week': 15,
    'day': 5,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Options strategy framing',
 'week': 15,
 'day': 5,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 05 Quiz

Topic: **Options strategy framing**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [16]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Options strategy framing')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Options strategy framing
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975


# Week 15 Day 06: Revision Sprint

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours across recall, rebuild, rerun, and retest blocks.

## 6-10 Hour Daily Structure
- **Core Block 1 (60 min):** Recall formulas, notation, and units without notes.
- **Core Block 2 (75 min):** Rebuild one weekday workflow from memory and compare with reference output.
- **Core Block 3 (60 min):** Rerun at least two notebooks from a clean kernel and capture mismatches.
- **Core Block 4 (60 min):** Diagnose root causes and update safeguards in code/comments.
- **Core Block 5 (45 min):** Interview drill, quiz, and escalation-rule rehearsal.
- **Required Extension Block A (60 min):** Re-test on one alternate ticker universe and one stress window.
- **Required Extension Block B (60 min):** Produce a remediation log with before/after evidence.
- **Optional Deep Work (0-4 hours):** Add robustness tests, monitor drift controls, and harden reporting.

## Why It Matters in Quant
Revision day converts fragile understanding into deployment-ready reliability by forcing recall, rebuild, and guardrail validation.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 05: Options strategy framing
- Previous lesson file: content/week-15/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 15 Day 07: Portfolio Project
- Next lesson file: content/week-15/day-07.md

## Revision Plan
- 90 minutes: active recall of weekday concepts and manual formula rewrite.
- 90 minutes: rebuild one weekday code workflow from memory.
- 90 minutes: restart kernel and rerun at least two day notebooks end-to-end.
- 90 minutes: error-log triage, retest plan, and guardrail refinement.
- Optional 0-4 hours: deepen weak areas with extra interview drill and code hardening.

## Focus Areas
- Redraw key payoff and Greek intuition charts
- Re-run one hedging simulation
- Review assumptions in volatility-surface interpretation

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Rebuild two weekday workflows and compare decision outputs after a clean rerun.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Revision Sprint'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Rebuild one workflow from memory, rerun from clean kernel, and document one corrected failure mode.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive two key formulas from this week with units and constraints.
2. Rebuild one weekday notebook workflow from memory and verify output parity.
3. Replay one stress regime and document one changed decision rule.
4. Add one extra validation test that would have caught a prior mistake.
5. Record a 60-second explanation of one corrected failure mode.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 06): Explain Portfolio Return and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use Portfolio Return as a decision bridge from market observations to position sizing. The formula is $\mu_p=w^\top\mu$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Revision Sprint' matter in live trading systems?
   - Model answer: "Revision Sprint matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Revision Sprint in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which failure mode still appears after reruns, and what concrete control will prevent it next week?

## Completion Checklist
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Two notebooks rerun from fresh kernel
- [ ] Next-week risk list prepared
- [ ] Interview drill recorded with one fallback action


## Week 15 Day 06 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [17]:
# Revision sprint demo: rebuild weekly core diagnostics
np.random.seed(1506)
prices = load_market_prices(['SPY', 'QQQ', 'AAPL'], start='2018-01-01')
returns = prices.pct_change().dropna()

summary = pd.DataFrame({
    'annual_return': returns.mean() * 252,
    'annual_vol': returns.std() * np.sqrt(252),
    'max_drawdown': (prices / prices.cummax() - 1).min(),
})
summary['sharpe_proxy'] = (summary['annual_return'] - 0.03) / summary['annual_vol'].replace(0, np.nan)
summary = summary.sort_values('sharpe_proxy', ascending=False)

print('Revision diagnostics (best risk-adjusted first):')
summary.round(4)


Revision diagnostics (best risk-adjusted first):


,annual_return,annual_vol,max_drawdown,sharpe_proxy
Ticker,,,,
AAPL,0.277000,0.306000,-0.385200,0.807300
QQQ,0.205600,0.238800,-0.351200,0.735400
SPY,0.151700,0.193100,-0.337200,0.630400


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [18]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10506)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Revision Sprint',
    'week': 15,
    'day': 6,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Revision Sprint',
 'week': 15,
 'day': 6,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 06 Quiz

Topic: **Revision Sprint**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [19]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Revision Sprint')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Revision Sprint
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975


# Week 15 Day 07: Portfolio Project

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours for implementation, validation, and communication drills.

## 6-10 Hour Daily Structure
- **Core Block 1 (60 min):** Restate objective, assumptions, and measurable success criteria.
- **Core Block 2 (75 min):** Build/clean data pipeline and verify timestamp integrity.
- **Core Block 3 (75 min):** Implement project logic and validate formulas against spot checks.
- **Core Block 4 (60 min):** Produce diagnostics, stress checks, and fallback pathways.
- **Core Block 5 (45 min):** Deliver interview-style defense with risk controls and escalation.
- **Required Extension Block A (60 min):** Re-run project on alternate assumptions and compare drift.
- **Required Extension Block B (60 min):** Prepare production memo and launch/no-launch decision log.
- **Optional Deep Work (0-4 hours):** Expand tests, improve monitoring, and polish stakeholder narrative.

## Why It Matters in Quant
Project day is where research quality meets execution discipline and communication quality under risk constraints.

## Continuity and Handoff
- Previous checkpoint: Week 15 Day 06: Revision Sprint
- Previous lesson file: content/week-15/day-06.md
- Today's deliverable: Options risk intuition notebook
- Next handoff target: Week 16 Day 01: Risk framework architecture
- Next lesson file: content/week-16/day-01.md

## Project Blueprint
### Project Title
Options risk intuition notebook

### Problem Statement
Build an options analytics notebook covering payoffs, Greeks, and hedging outcomes.

### Data Sources
- Synthetic option chain
- Underlying paths
- Volatility scenarios

### Implementation Steps
1. Implement payoff analytics
2. Estimate and visualize Greeks
3. Run hedging simulation
4. Compare strategy scenarios
5. Summarize practical risk insights

### Evaluation Metrics
- Hedge error
- Scenario PnL spread
- Risk clarity
- Interpretability

### Execution Standard
- [ ] Notebook/script runs from clean start without hidden state
- [ ] Outputs include at least one diagnostic table and one chart
- [ ] One explicit risk guardrail and fallback action are documented

### Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-02 to 2020-04, 2022-01 to 2022-10
- Scenario context: cross-asset drawdown with correlation spikes
- Day objective: Deliver a capstone-quality notebook and summarize one trade-off under stress assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Portfolio Project'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Ship the project notebook with reproducible outputs, controls, and one escalation rule.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. State project objective and one hard failure condition in under 45 seconds.
2. Validate one formula output against a manual spot-check.
3. Show one stress scenario where your decision changes and explain why.
4. Add one edge-case test and one fallback rule to the notebook.
5. Deliver a one-minute PM summary with one risk guardrail.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 15 Day 07): Explain Portfolio Variance and define every symbol clearly for a cross-asset drawdown week with correlation spikes.
   - Model answer: "I use Portfolio Variance as a decision bridge from market observations to position sizing. The formula is $\sigma_p^2=w^\top\Sigma w$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then force rebalance when risk-budget contribution of one sleeve exceeds 40%. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify covariance window consistency, stale-price filtering, and exposure normalization checks. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Portfolio Project' matter in live trading systems?
   - Model answer: "Portfolio Project matters because portfolio choices are constrained by covariance instability, transaction costs, and risk budgets. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I de-risk to benchmark weights, rerun stress tests, and re-enable risk only after guardrails pass. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Portfolio Project in an investment committee session after correlation breakdown."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
What single risk control would block launch today, and what evidence would clear it?

## Completion Checklist
- [ ] Project notebook runs cleanly from fresh kernel
- [ ] Risk guardrail and fallback action documented
- [ ] Stress scenario comparison completed
- [ ] One-page summary memo finalized
- [ ] Launch/no-launch decision recorded with evidence


## Week 15 Day 07 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [20]:
# Project day demo: mini portfolio report with trade recommendation
np.random.seed(1507)
assets = ['SPY', 'QQQ', 'TLT', 'GLD']
prices = load_market_prices(assets, start='2019-01-01')
returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_vol = returns.std() * np.sqrt(252)
score = (annual_return - 0.03) / annual_vol.replace(0, np.nan)

report = pd.DataFrame({
    'annual_return': annual_return,
    'annual_vol': annual_vol,
    'sharpe_proxy': score,
}).sort_values('sharpe_proxy', ascending=False)

top_asset = report.index[0]
print('Project summary:')
print(report.round(4))
print(f"\nSuggested focus asset for follow-up research: {top_asset}")


Project summary:
        annual_return  annual_vol  sharpe_proxy
Ticker                                         
GLD          0.194100    0.172700      0.950000
QQQ          0.232200    0.240200      0.841900
SPY          0.177800    0.196000      0.754000
TLT         -0.005100    0.162600     -0.215600

Suggested focus asset for follow-up research: GLD


## ReAct Verification Cell
Validate trade logic with a risk guardrail before reading the model quiz answers.

In [21]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(10507)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Portfolio Project',
    'week': 15,
    'day': 7,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Portfolio Project',
 'week': 15,
 'day': 7,
 'observe_annual_return': 0.14923378440213875,
 'observe_annual_vol': 0.2050127751352247,
 'reason_sharpe_proxy': 0.5815919731026183,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 15 Day 07 Quiz

Topic: **Portfolio Project**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Portfolio Return (mu_p = w^T * mu) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: rebalance when any sleeve exceeds its risk-budget threshold.
Hint for Q3: covariance-window consistency and stale-price filters
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [22]:
# Quiz model answers (auto-generated)
w = np.array([0.55, 0.30, 0.15])
mu = np.array([0.12, 0.08, 0.05])
metric_value = float(w @ mu)
metric_name = 'portfolio_expected_return'

print('Interview Question 1 (model answer):')
print('  Formula focus: Portfolio Return')
print('  Formula text : mu_p = w^T * mu')
print('  Scenario     : cross-asset drawdown phase')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  rebalance when one sleeve contributes over 40% of risk.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Portfolio Project')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: move to defensive benchmark weights and rerun stress tests.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Portfolio Return
  Formula text : mu_p = w^T * mu
  Scenario     : cross-asset drawdown phase
  Numeric value: 0.0975

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  rebalance when one sleeve contributes over 40% of risk.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Portfolio Project
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: move to defensive benchmark weights and rerun stress tests.

Numeric verification:
  metric_name  = portfolio_expected_return
  metric_value = 0.0975
